# 09 — External Validation: MIMIC-IV-ECG (STEMI-proxy qua ICD-10)

**Dự án ACS-ECG-AI (Vinmec)** · nối tiếp [`08_stemi_stability_calibration.ipynb`](08_stemi_stability_calibration.ipynb)

## Mục đích

Notebook `08` chốt model **Average Ensemble (8 kiến trúc)** và ngưỡng vận hành trên TEST giữ riêng
của **chính ACS-ECG 2026** — cùng nguồn thu thập, cùng quy trình gán nhãn với tập train. Notebook này
kiểm tra model đó trên một **nguồn dữ liệu độc lập hoàn toàn** — MIMIC-IV-ECG (Mỹ, BIDMC Boston) — để
trả lời câu hỏi: *model có generalize ra ngoài phân bố đã học, hay chỉ đang khớp với đặc thù của
ACS-ECG 2026?*

## ⚠️ Bản chất nhãn — đọc trước khi diễn giải kết quả

MIMIC-IV-ECG (bản waveform) **không có nhãn chẩn đoán**. Nhãn STEMI/non-STEMI ở đây được suy ra từ
**mã ICD-10 chẩn đoán ra viện** (qua MIMIC-IV-ECG-Ext-ICD), không phải do bác sĩ tim mạch đọc trực
tiếp ECG tại thời điểm đó như ACS-ECG 2026 (xác nhận qua chụp mạch vành DSA). Vì vậy:

- Đây là **"STEMI-proxy qua ICD"**, không phải ground-truth ngang hàng với ACS-ECG 2026.
- Một nghiên cứu phương pháp luận độc lập kết luận: *"nếu không có thêm điều chỉnh, mã ICD-10 không
  đủ để phân biệt rõ ràng loại nhồi máu cơ tim cấp"* — kết quả ở đây nên được đọc như **kiểm chứng bổ
  sung mang tính định hướng**, không thay thế vai trò của TEST nội bộ trong notebook `08`.
- Bộ mã dùng để xác định STEMI: `I21.0–I21.3, I22.0, I22.1, I22.8`. NSTEMI: `I21.4`.

## Thiết kế lấy mẫu — vì sao không lấy ngẫu nhiên 10% toàn bộ

STEMI vốn hiếm (nhóm I21 acute MI thậm chí không nằm trong 4 nhóm bệnh tim mạch phổ biến nhất của
MIMIC-IV-ECG-ICD-ED). Lấy ngẫu nhiên 10% toàn bộ ~800.000 bản ghi có nguy cơ **gần như không còn ca
STEMI dương nào** trong mẫu. Thay vào đó, notebook này:

1. Lọc nhãn (ICD-10) **trước**, trên toàn bộ bảng metadata (nhẹ, không cần tải waveform).
2. Giữ **100% ca dương (STEMI)** tìm được.
3. Chỉ lấy mẫu ~10% (có thể chỉnh `NEG_SAMPLE_FRAC`) trong nhóm **âm rõ** (không có bất kỳ mã MI nào).
4. Chỉ tải waveform (.hea/.dat) đúng những bản ghi đã chọn — không tải nguyên khối 33,8 GB.

Cách này vừa đúng tinh thần "tải ~10% để thử nghiệm trước", vừa đảm bảo tập test còn đủ ca dương để
tính Sensitivity/AUPRC có ý nghĩa.


## 1. Cài thư viện

In [49]:
!pip install -q wfdb requests
import importlib.util
import os
import sys
import requests
print("Python:", sys.version.split()[0])


Python: 3.13.15


## 2. Cấu hình

Đường dẫn khớp với các checkpoint/OOF anh đã cung cấp. **Sửa `DRIVE_PROJECT`/`EXT_ICD_ZIP_NAME`
nếu khác thực tế.**


In [50]:
import os
from pathlib import Path
import torch

IS_COLAB = importlib.util.find_spec("google.colab") is not None
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/ACS-ECG-AI")   # <<< SỬA nếu đặt tên khác
PERSIST_DIR   = DRIVE_PROJECT / "outputs"

# ---- Đường dẫn anh đã cung cấp ----
KFOLD_MODEL_DIR   = PERSIST_DIR / "models" / "stemi_kfold_holdout"   # checkpoint FINAL (8 kiến trúc) + fold
COMPARE_MODEL_DIR = PERSIST_DIR / "models" / "stemi_compare"         # notebook 03, chỉ đọc để đối chiếu
OOF_NPZ_PATH      = PERSIST_DIR / "oof" / "oof_STEMI_full_k5r1_holdout.npz"
ACS_CACHE_DIR     = PERSIST_DIR / "cache"                            # cache tín hiệu ACS-ECG 2026 đã lọc

# ---- Dữ liệu gốc ACS-ECG 2026 (để khôi phục POOL_IDX + mean_pool/std_pool giống hệt notebook 08) ----
DRIVE_DATA_ZIP = DRIVE_PROJECT / "datasets.zip"          # <<< SỬA nếu tên zip khác
ACS_DATA_ROOT  = Path("/content/datasets_acs")           # giải nén cục bộ để đọc train.csv

# ---- MIMIC-IV-ECG-Ext-ICD: anh tự tải (credentialed) và để dạng .zip ngay trong ACS-ECG-AI ----
EXT_ICD_ZIP_NAME = "mimic-iv-ecg-ext-icd-labels.zip"      # <<< SỬA nếu đặt tên file khác
EXT_ICD_ZIP_PATH = DRIVE_PROJECT / EXT_ICD_ZIP_NAME

# ---- Nơi lưu output của notebook này ----
EXT_WORK_DIR  = Path("/content/mimic_ext")                       # làm việc tạm trên đĩa local Colab
EXT_PERSIST   = PERSIST_DIR / "external_validation" / "mimic_icd"  # bền qua các phiên, lưu về Drive
EXT_WORK_DIR.mkdir(parents=True, exist_ok=True)
EXT_PERSIST.mkdir(parents=True, exist_ok=True)

# ---- Tham số mô hình / tín hiệu -- PHẢI khớp notebook 08 ----
SEED = 42
FS, SIGNAL_LEN, NUM_LEADS = 500, 5000, 12       # 12 chuyển đạo x 10s @ 500Hz
BP_LOW, BP_HIGH, BP_ORDER = 0.5, 40.0, 3        # bandpass Butterworth -- giống notebook 08
TARGET_LABEL = "STEMI"
CANDIDATES = ["PlainCNN", "XResNet1D", "ConvNeXtV2_1D", "AiTiAMI", "ResNet1D",
              "SEResNet1D", "CNN+BiLSTM", "InceptionTime1D"]   # 8 ứng viên, giống notebook 08
TARGET_SENS_PRIMARY = 0.91    # chính sách ngưỡng STEMI đã dùng trong báo cáo (khớp mục 23a của notebook 08)

# ---- Tham số lấy mẫu MIMIC ----
NEG_SAMPLE_FRAC = 0.10   # tỉ lệ lấy mẫu trong nhóm ÂM RÕ (không mã MI nào) -- xem giải thích ở trên
MAX_NEG_SAMPLES = 4000   # chặn trên, tránh tải quá nhiều nếu pool âm quá lớn
RANDOM_STATE = SEED

GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if GPU_AVAILABLE else "cpu")
BATCH_SIZE = 64 if GPU_AVAILABLE else 16

print("Thiết bị        :", DEVICE)
print("Checkpoint FINAL:", KFOLD_MODEL_DIR)
print("OOF             :", OOF_NPZ_PATH)
print("Cache ACS-ECG   :", ACS_CACHE_DIR)
print("Ext-ICD zip     :", EXT_ICD_ZIP_PATH, "--", "TỒN TẠI" if EXT_ICD_ZIP_PATH.exists() else "!! KHÔNG THẤY")
print("Output ngoài    :", EXT_PERSIST)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thiết bị        : cuda
Checkpoint FINAL: /content/drive/MyDrive/ACS-ECG-AI/outputs/models/stemi_kfold_holdout
OOF             : /content/drive/MyDrive/ACS-ECG-AI/outputs/oof/oof_STEMI_full_k5r1_holdout.npz
Cache ACS-ECG   : /content/drive/MyDrive/ACS-ECG-AI/outputs/cache
Ext-ICD zip     : /content/drive/MyDrive/ACS-ECG-AI/mimic-iv-ecg-ext-icd-labels.zip -- TỒN TẠI
Output ngoài    : /content/drive/MyDrive/ACS-ECG-AI/outputs/external_validation/mimic_icd


## 3. Tám kiến trúc model

Copy **nguyên văn** từ notebook `08` (mục 12) — bắt buộc giống hệt để `load_state_dict` không lỗi
lệch kiến trúc. Không sửa bất kỳ tham số nào ở đây.


In [51]:
import torch
import torch.nn as nn


# ---------------------------------------------------------------- 1. PlainCNN
class PlainCNN(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, c_in = [], NUM_LEADS
        for c_out in channels:
            layers += [nn.Conv1d(c_in, c_out, 7, padding=3, bias=False),
                       nn.BatchNorm1d(c_out), nn.ReLU(inplace=True), nn.MaxPool1d(4)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(-1)


# ---------------------------------------------------------------- 2. ResNet1D
class ResidualBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        return self.relu(self.bn2(self.conv2(out)) + idt)


class ResNet1D(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(ResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 3. InceptionTime1D
class InceptionModule(nn.Module):
    def __init__(self, c_in, n_filters=32, kernels=(39, 19, 9), bottleneck=32):
        super().__init__()
        self.bottleneck = nn.Conv1d(c_in, bottleneck, 1, bias=False)
        self.convs = nn.ModuleList(
            [nn.Conv1d(bottleneck, n_filters, k, padding=k // 2, bias=False) for k in kernels])
        self.pool_conv = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1),
                                       nn.Conv1d(c_in, n_filters, 1, bias=False))
        self.bn = nn.BatchNorm1d(n_filters * (len(kernels) + 1))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        b = self.bottleneck(x)
        return self.relu(self.bn(torch.cat([c(b) for c in self.convs] + [self.pool_conv(x)], 1)))


class InceptionTime1D(nn.Module):
    def __init__(self, n_filters=32, depth=6):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 4, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True))
        c_out = n_filters * 4
        self.blocks = nn.ModuleList()
        self.shortcuts = nn.ModuleList()
        self.pools = nn.ModuleList()
        c_in = 32
        res_c = 32
        for d in range(depth):
            self.blocks.append(InceptionModule(c_in, n_filters))
            if d % 3 == 2:
                self.shortcuts.append(nn.Sequential(nn.Conv1d(res_c, c_out, 1, bias=False),
                                                    nn.BatchNorm1d(c_out)))
                self.pools.append(nn.MaxPool1d(4))
                res_c = c_out
            else:
                self.shortcuts.append(None)
                self.pools.append(None)
            c_in = c_out
        self.relu = nn.ReLU(inplace=True)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_out, 1))

    def forward(self, x):
        x = self.stem(x)
        res = x
        for blk, short, pool in zip(self.blocks, self.shortcuts, self.pools):
            x = blk(x)
            if short is not None:
                x = self.relu(x + short(res))
                x = pool(x)
                res = x
        return self.head(x).squeeze(-1)


# ---------------------------------------------------------------- 4. CNN + BiLSTM
class CNNBiLSTM(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        layers, c_in = [], NUM_LEADS
        for c_out in (32, 64, 128):
            layers += [nn.Conv1d(c_in, c_out, 7, padding=3, bias=False),
                       nn.BatchNorm1d(c_out), nn.ReLU(inplace=True), nn.MaxPool1d(4)]
            c_in = c_out
        self.cnn = nn.Sequential(*layers)
        self.lstm = nn.LSTM(c_in, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden * 2, 1))

    def forward(self, x):
        h = self.cnn(x).transpose(1, 2)
        out, _ = self.lstm(h)
        return self.head(out.mean(dim=1)).squeeze(-1)


# ---------------------------------------------------------------- 5. SEResNet1D
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(channels, hidden), nn.ReLU(inplace=True),
                                nn.Linear(hidden, channels), nn.Sigmoid())

    def forward(self, x):
        w = self.fc(self.pool(x).squeeze(-1)).unsqueeze(-1)
        return x * w


class SEResidualBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.se = SEBlock1D(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        out = self.se(self.bn2(self.conv2(out)))
        return self.relu(out + idt)


class SEResNet1D(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(SEResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 6. XResNet1D
class XResBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.act = nn.SiLU(inplace=True)
        if stride == 1 and c_in == c_out:
            self.short = nn.Identity()
        else:
            self.short = nn.Sequential(
                nn.AvgPool1d(stride, ceil_mode=True), nn.Conv1d(c_in, c_out, 1, bias=False),
                nn.BatchNorm1d(c_out))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.act(self.bn1(self.conv1(x))))
        out = self.bn2(self.conv2(out))
        return self.act(out + idt)


class XResNet1D(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(NUM_LEADS, 32, 5, 2, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.Conv1d(32, 32, 5, 1, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.Conv1d(32, 32, 5, 1, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(XResBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.SiLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 7. ConvNeXtV2_1D
class GRN1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, 1, channels))
        self.beta = nn.Parameter(torch.zeros(1, 1, channels))

    def forward(self, x):
        gx = torch.norm(x, p=2, dim=1, keepdim=True)
        nx = gx / (gx.mean(dim=-1, keepdim=True) + 1e-6)
        return self.gamma * (x * nx) + self.beta + x


class ConvNeXtV2Block1D(nn.Module):
    def __init__(self, channels, expand=4, dropout=0.1):
        super().__init__()
        self.dwconv = nn.Conv1d(channels, channels, 7, padding=3, groups=channels, bias=False)
        self.norm = nn.LayerNorm(channels)
        self.pw1 = nn.Linear(channels, channels * expand)
        self.act = nn.GELU()
        self.grn = GRN1D(channels * expand)
        self.pw2 = nn.Linear(channels * expand, channels)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        idt = x
        x = self.dwconv(x).transpose(1, 2)
        x = self.norm(x)
        x = self.pw2(self.grn(self.act(self.pw1(x))))
        x = self.drop(x).transpose(1, 2)
        return x + idt


class ConvNeXtV2Downsample1D(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.norm = nn.BatchNorm1d(c_in)
        self.conv = nn.Conv1d(c_in, c_out, 2, stride=2, bias=False)

    def forward(self, x):
        return self.conv(self.norm(x))


class ConvNeXtV2_1D(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), depths=(1, 1, 2, 1), dropout=0.1):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, channels[0], 4, stride=4, bias=False),
                                  nn.BatchNorm1d(channels[0]))
        stages, c_in = [], channels[0]
        for stage_i, (c_out, depth) in enumerate(zip(channels, depths)):
            if stage_i > 0:
                stages.append(ConvNeXtV2Downsample1D(c_in, c_out))
            stages += [ConvNeXtV2Block1D(c_out, dropout=dropout) for _ in range(depth)]
            c_in = c_out
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.LayerNorm(c_in), nn.Dropout(dropout), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.stages(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 8. AiTiAMI
class AiTiAMIBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=5, stride=2, dropout=0.15):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        return self.relu(self.bn2(self.conv2(out)) + idt)


class AiTiAMI(nn.Module):
    def __init__(self, channels=(48, 96, 192, 256, 320), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 48, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(48), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 48
        for i, c_out in enumerate(channels):
            blocks.append(AiTiAMIBlock1D(c_in, c_out, stride=2 if i < 4 else 1))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.maxpool = nn.AdaptiveMaxPool1d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(c_in * 2, 128), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, x):
        feat = self.blocks(self.stem(x))
        pooled = torch.cat([self.avgpool(feat), self.maxpool(feat)], dim=1)
        return self.head(pooled).squeeze(-1)


MODELS = {
    "PlainCNN": PlainCNN, "ResNet1D": ResNet1D, "InceptionTime1D": InceptionTime1D,
    "CNN+BiLSTM": CNNBiLSTM, "SEResNet1D": SEResNet1D, "XResNet1D": XResNet1D,
    "ConvNeXtV2_1D": ConvNeXtV2_1D, "AiTiAMI": AiTiAMI,
}
print("Đã định nghĩa", len(MODELS), "kiến trúc:", ", ".join(MODELS.keys()))


Đã định nghĩa 8 kiến trúc: PlainCNN, ResNet1D, InceptionTime1D, CNN+BiLSTM, SEResNet1D, XResNet1D, ConvNeXtV2_1D, AiTiAMI


## 4. Khôi phục chuẩn hoá (mean/std) và ngưỡng vận hành từ ACS-ECG 2026

Checkpoint FINAL chỉ lưu `state_dict`, **không lưu** `mean_pool`/`std_pool` dùng để z-score lúc train.
Phải tính lại đúng như notebook `08` (mục 14c): `norm_stats(POOL_IDX)` trên **đúng tập POOL** (85%
bệnh nhân, sau khi tách 15% TEST). Nếu chuẩn hoá sai, xác suất đầu ra sẽ lệch hệ thống dù model đúng.

Cần `train.csv` gốc (trong `datasets.zip`) để tái tạo đúng `POOL_IDX` — **cùng `SEED=42`,
`TEST_SIZE=0.15`, cùng logic chia theo bệnh nhân** như notebook 08. Sau đó dùng **cache tín hiệu đã
có sẵn trên Drive** (không tính lại từ đầu, chỉ cần thống kê mean/std).


In [52]:
import hashlib
import shutil
import zipfile

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

if not ACS_DATA_ROOT.exists() or not any(ACS_DATA_ROOT.rglob("train.csv")):
    print("Giải nén train.csv từ datasets.zip ...")
    ACS_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_DATA_ZIP) as zf:
        csv_members = [n for n in zf.namelist() if n.lower().endswith("train.csv")]
        assert csv_members, "Không tìm thấy train.csv trong datasets.zip"
        zf.extract(csv_members[0], ACS_DATA_ROOT)
    print("Xong.")

_train_csv = next(ACS_DATA_ROOT.rglob("train.csv"))
df_raw = pd.read_csv(_train_csv)
df_raw["record_stem"] = df_raw["ecg_row_record"].astype(str).str.replace(".dat", "", regex=False)
COL_PATIENT = "Patient_id"

df_all = df_raw.copy()
df_all[TARGET_LABEL] = df_all["STEMI"].astype(int)
df_acs = df_all.reset_index(drop=True)          # RUN_MODE="full" trong notebook 08 -> df == df_all
y_acs = df_acs[TARGET_LABEL].values.astype(np.float32)

# ---- Tái tạo ĐÚNG POOL/TEST split của notebook 08 (mục 12b) ----
TEST_SIZE = 0.15
pat_all = df_acs.groupby(COL_PATIENT)[TARGET_LABEL].max().reset_index()
pat_pool, pat_test = train_test_split(pat_all, test_size=TEST_SIZE,
                                      stratify=pat_all[TARGET_LABEL], random_state=SEED + 200)
TEST_PATIENTS = set(pat_test[COL_PATIENT])
IS_TEST = df_acs[COL_PATIENT].isin(TEST_PATIENTS).to_numpy()
POOL_IDX = df_acs.index[~IS_TEST].to_numpy()
TEST_IDX = df_acs.index[IS_TEST].to_numpy()
assert not (set(df_acs.loc[POOL_IDX, COL_PATIENT]) & TEST_PATIENTS), "RÒ RỈ: bệnh nhân TEST lọt vào POOL"
print(f"POOL: {len(POOL_IDX):,} bản ghi | TEST: {len(TEST_IDX):,} bản ghi "
      f"(khớp lại đúng split của notebook 08)")

# ---- Định vị cache đã build sẵn (hash giống hệt logic notebook 08, mục 7) ----
records = df_acs["record_stem"].tolist()
_hash = hashlib.md5("|".join(records).encode()).hexdigest()[:8]
ACS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_cache_name = f"sig_full_n{len(df_acs)}_{_hash}.npy"
_cache_candidates = list(ACS_CACHE_DIR.glob(_cache_name))

if _cache_candidates:
    CACHE_ACS = np.load(_cache_candidates[0], mmap_mode="r")
    print("Cache ACS-ECG 2026 (có sẵn):", CACHE_ACS.shape, CACHE_ACS.dtype,
          "--", _cache_candidates[0].name)
else:
    # ---- KHÔNG tìm thấy cache khớp hash -- tự build lại từ raw signal, giống hệt notebook 08 ----
    print(f"!! Không tìm thấy cache khớp ({_cache_name}) tại {ACS_CACHE_DIR}")
    print("   -> Tự build lại cache từ raw signal (giống notebook 08, mục 7). Có thể mất một lúc.\n")

    import time

    import wfdb
    from scipy.signal import butter, filtfilt

    def _find_dir(root, names):
        wanted = {n.lower() for n in names}
        if not root.exists():
            return None
        for c in sorted(root.iterdir()):
            if c.is_dir() and c.name.lower() in wanted:
                return c
        for c in root.rglob("*"):
            if c.is_dir() and c.name.lower() in wanted:
                return c
        return None

    RAW_DIR = _find_dir(ACS_DATA_ROOT, ["row_data", "raw_data"])
    if RAW_DIR is None or not any(RAW_DIR.glob("*.dat")):
        # train.csv trước đó chỉ trích riêng, chưa có raw signal -- giải nén thêm phần row_data/raw_data
        print("   Giải nén thêm thư mục dữ liệu thô (row_data/raw_data) từ datasets.zip ...")
        with zipfile.ZipFile(DRIVE_DATA_ZIP) as zf:
            _names = zf.namelist()
            _raw_prefix = next(
                (n.split("/")[0] + "/" for n in _names
                 if "row_data" in n.lower() or "raw_data" in n.lower()), None)
            assert _raw_prefix, (
                "Không tìm thấy thư mục row_data/raw_data trong datasets.zip -- kiểm tra lại "
                "cấu trúc file zip hoặc trỏ ACS_DATA_ROOT sang nơi đã giải nén sẵn toàn bộ dataset.")
            _members = [n for n in _names if n.startswith(_raw_prefix)]
            print(f"   Đang giải nén {len(_members):,} file ...")
            zf.extractall(ACS_DATA_ROOT, members=_members)
        RAW_DIR = _find_dir(ACS_DATA_ROOT, ["row_data", "raw_data"])
        assert RAW_DIR is not None, "Vẫn không tìm thấy row_data/raw_data sau khi giải nén."
    print("   RAW_DIR:", RAW_DIR)

    _B, _A = butter(BP_ORDER, [BP_LOW / (FS / 2), BP_HIGH / (FS / 2)], btype="band")
    _TRUNCATED = []

    def _load_signal(stem: str) -> np.ndarray:
        path = str(RAW_DIR / stem)
        try:
            rec = wfdb.rdrecord(path)
        except ValueError:                     # .dat ngắn hơn header khai báo
            n_sig = int((RAW_DIR / f"{stem}.hea").read_text().splitlines()[0].split()[1])
            n = (RAW_DIR / f"{stem}.dat").stat().st_size // (n_sig * 2)
            rec = wfdb.rdrecord(path, sampto=n)
            _TRUNCATED.append(stem)
        return np.asarray(rec.p_signal, dtype=np.float32).T

    def _preprocess(sig: np.ndarray) -> np.ndarray:
        sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
        if sig.shape[1] < SIGNAL_LEN:
            sig = np.pad(sig, ((0, 0), (0, SIGNAL_LEN - sig.shape[1])))
        return np.ascontiguousarray(filtfilt(_B, _A, sig[:, :SIGNAL_LEN], axis=1), dtype=np.float32)

    _cache_path = ACS_CACHE_DIR / _cache_name
    arr = np.lib.format.open_memmap(_cache_path, mode="w+", dtype=np.float16,
                                    shape=(len(records), NUM_LEADS, SIGNAL_LEN))
    t0 = time.time()
    step = max(1, len(records) // 8)
    for i, stem in enumerate(records):
        arr[i] = _preprocess(_load_signal(stem)).astype(np.float16)
        if (i + 1) % step == 0 or i + 1 == len(records):
            arr.flush()
            el = max(time.time() - t0, 1e-6)
            print(f"     {i + 1:>6,}/{len(records):,}  {(i + 1) / el:5.0f} rec/s  "
                  f"ETA {(len(records) - i - 1) / ((i + 1) / el):4.0f}s")
    del arr
    print(f"   Xong trong {time.time() - t0:.0f}s -- đã lưu {_cache_path}")
    if _TRUNCATED:
        print(f"   Bản ghi bị cắt ngắn, đã zero-pad: {_TRUNCATED}")

    CACHE_ACS = np.load(_cache_path, mmap_mode="r")
    print("\nCache ACS-ECG 2026 (mới build):", CACHE_ACS.shape, CACHE_ACS.dtype, "--", _cache_path.name)


def norm_stats(idx, cache, chunk=256):
    order = np.sort(np.asarray(idx))
    s = np.zeros(NUM_LEADS); ss = np.zeros(NUM_LEADS); cnt = 0
    for i in range(0, len(order), chunk):
        b = np.asarray(cache[order[i:i + chunk]], dtype=np.float64)
        s += b.sum(axis=(0, 2)); ss += (b ** 2).sum(axis=(0, 2)); cnt += b.shape[0] * b.shape[2]
    m = s / cnt
    return m.astype(np.float32), np.sqrt(np.maximum(ss / cnt - m ** 2, 1e-12)).astype(np.float32)


mean_pool, std_pool = norm_stats(POOL_IDX, CACHE_ACS)
print("mean_pool:", np.round(mean_pool, 4))
print("std_pool :", np.round(std_pool, 4))


POOL: 15,279 bản ghi | TEST: 2,681 bản ghi (khớp lại đúng split của notebook 08)
Cache ACS-ECG 2026 (có sẵn): (17960, 12, 5000) float16 -- sig_full_n17960_c20c6dba.npy
mean_pool: [-0.0001  0.0004  0.0002  0.0001 -0.0004  0.0004 -0.0002  0.      0.0003
  0.0002  0.0001  0.0002]
std_pool : [0.4318 0.4686 0.4878 0.4612 0.4632 0.4855 0.5005 0.5792 0.5563 0.5456
 0.6324 0.7319]


## 5. Khôi phục ngưỡng vận hành từ OOF (Average Ensemble, Sensitivity ≥ 91%)

Đúng phương pháp luận notebook 08: **chốt ngưỡng từ OOF trên POOL, áp nguyên xi lên tập ngoài** — không
refit trên MIMIC. Dùng `Average-CV` (trung bình xác suất 8 model) vì đây là model tốt nhất hiện tại
theo anh xác nhận.


In [53]:
OOF = np.load(OOF_NPZ_PATH)
y_oof = OOF["y"]
OOF_P = {name: OOF[f"p_{name}"][0] for name in CANDIDATES}     # rep 0
VALID = np.isfinite(OOF_P[CANDIDATES[0]])
print(f"OOF phủ {VALID.sum():,}/{len(y_oof):,} bản ghi | {int(y_oof[VALID].sum())} ca dương")

OOF_AVG = np.mean([OOF_P[n] for n in CANDIDATES], axis=0)


def threshold_at_sensitivity(y_true, y_prob, target_sens):
    """Giống hệt notebook 08 (mục 17b): ngưỡng LỚN NHẤT sao cho Sensitivity thực tế >= target."""
    y_true = np.asarray(y_true).astype(int)
    pos_scores = np.sort(y_prob[y_true == 1])[::-1]
    n_pos = len(pos_scores)
    k = min(int(np.ceil(target_sens * n_pos)), n_pos)
    return float(pos_scores[k - 1]) if k > 0 else 1.0


THRESHOLD = threshold_at_sensitivity(y_oof[VALID], OOF_AVG[VALID], TARGET_SENS_PRIMARY)
print(f"Ngưỡng Average Ensemble @ Sensitivity >= {TARGET_SENS_PRIMARY:.0%} (chốt từ POOL OOF): "
      f"{THRESHOLD:.4f}")


OOF phủ 15,279/17,960 bản ghi | 1225 ca dương
Ngưỡng Average Ensemble @ Sensitivity >= 91% (chốt từ POOL OOF): 0.2417


## 6. Giải nén và nạp bảng nhãn MIMIC-IV-ECG-Ext-ICD

Đọc trực tiếp từ file zip anh đã tải thủ công (credentialed) và để trong thư mục `ACS-ECG-AI` trên
Drive — không tải lại qua mạng, không cần xác thực trong notebook này.


In [54]:
import zipfile

assert EXT_ICD_ZIP_PATH.exists(), (
    f"Không thấy {EXT_ICD_ZIP_PATH} -- kiểm tra lại EXT_ICD_ZIP_NAME ở mục 2, hoặc đường dẫn "
    f"nơi anh đã tải file mimic-iv-ecg-ext-icd-labels.zip lên Drive.")

EXT_ICD_EXTRACT_DIR = EXT_WORK_DIR / "ext_icd_raw"
if not any(EXT_ICD_EXTRACT_DIR.glob("**/records_w_diag_icd10.csv")):
    print("Giải nén", EXT_ICD_ZIP_PATH.name, "...")
    EXT_ICD_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(EXT_ICD_ZIP_PATH) as zf:
        zf.extractall(EXT_ICD_EXTRACT_DIR)
    print("Xong.")

_ext_icd_csv = next(EXT_ICD_EXTRACT_DIR.glob("**/records_w_diag_icd10.csv"))
df_icd = pd.read_csv(_ext_icd_csv)
print(f"{len(df_icd):,} dòng | cột: {list(df_icd.columns)}")
display(df_icd.head(3))


800,035 dòng | cột: ['file_name', 'study_id', 'subject_id', 'ecg_time', 'ed_stay_id', 'ed_hadm_id', 'hosp_hadm_id', 'ed_diag_ed', 'ed_diag_hosp', 'hosp_diag_hosp', 'all_diag_hosp', 'all_diag_all', 'gender', 'age', 'anchor_year', 'anchor_age', 'dod', 'ecg_no_within_stay', 'ecg_taken_in_ed', 'ecg_taken_in_hosp', 'ecg_taken_in_ed_or_hosp', 'fold', 'strat_fold']


,file_name,study_id,subject_id,ecg_time,ed_stay_id,ed_hadm_id,hosp_hadm_id,ed_diag_ed,ed_diag_hosp,hosp_diag_hosp,...,age,anchor_year,anchor_age,dod,ecg_no_within_stay,ecg_taken_in_ed,ecg_taken_in_hosp,ecg_taken_in_ed_or_hosp,fold,strat_fold
0,mimic-iv-ecg-diagnostic-electrocardiogram-matc...,40689238,10000032,2180-07-23 08:44:00,39399961.0,29079034.0,NaN,"['R4182', 'G9340']","['F319', 'J449', 'B182', 'E871', 'V462', 'I958...",[],...,52.0,2180.0,52.0,2180-09-09,0,True,False,True,17,9
1,mimic-iv-ecg-diagnostic-electrocardiogram-matc...,44458630,10000032,2180-07-23 09:54:00,39399961.0,29079034.0,NaN,"['R4182', 'G9340']","['F319', 'J449', 'B182', 'E871', 'V462', 'I958...",[],...,52.0,2180.0,52.0,2180-09-09,1,True,False,True,17,9
2,mimic-iv-ecg-diagnostic-electrocardiogram-matc...,49036311,10000032,2180-08-06 09:07:00,NaN,NaN,25742920.0,[],[],"['J449', 'B182', 'E871', 'R197', 'V462', 'R188...",...,52.0,2180.0,52.0,2180-09-09,0,False,True,True,17,9


## 7. Lọc STEMI / non-STEMI theo mã ICD-10

Cấu trúc cột đã xác nhận chính thức từ tài liệu PhysioNet (không còn phải đoán như bản trước):
`all_diag_all` = `all_diag_hosp` nếu có, ngược lại lấy `ed_diag_ed`; mã đã được chuẩn hoá về tối đa
5 ký tự, bỏ dấu chấm (ví dụ `I21.0` → `I210`).

Nhãn:
- **STEMI (dương)**: có ít nhất 1 mã trong `I21.0, I21.1, I21.2, I21.3, I22.0, I22.1, I22.8`
- **Non-STEMI rõ (âm)**: có ghi nhận chẩn đoán (không rỗng) nhưng **không** có bất kỳ mã MI nào khác
  (`I21.*`, `I22.*`, `I25.2` cũ) — loại các ca "không có chẩn đoán liên kết" ra khỏi tập test nhị phân
  để giảm nhiễu nhãn.


In [55]:
STEMI_CODES = {"I210", "I211", "I212", "I213", "I220", "I221", "I228"}
MI_RELATED_PREFIXES = ("I21", "I22", "I252")   # loại khỏi nhóm "âm rõ" nếu có bất kỳ mã MI nào khác

DIAG_COL = "all_diag_all"        # xác nhận chính thức trong tài liệu dataset
assert DIAG_COL in df_icd.columns, f"Không thấy cột {DIAG_COL} -- kiểm tra lại version dataset."


def _codes_of(cell):
    """Chuẩn hoá 1 dòng (chuỗi mã cách nhau bởi dấu phẩy/khoảng trắng, hoặc list-like) -> set mã."""
    if pd.isna(cell):
        return set()
    s = str(cell)
    for ch in "[]'\"":
        s = s.replace(ch, "")
    parts = [p.strip().upper().replace(".", "") for p in s.replace(";", ",").split(",")]
    return {p for p in parts if p}


df_icd["_codes"] = df_icd[DIAG_COL].apply(_codes_of)
df_icd["_is_stemi"] = df_icd["_codes"].apply(lambda cs: bool(cs & STEMI_CODES))
df_icd["_has_any_mi"] = df_icd["_codes"].apply(
    lambda cs: any(c.startswith(MI_RELATED_PREFIXES) for c in cs))
df_icd["_has_any_diag"] = df_icd["_codes"].apply(lambda cs: len(cs) > 0)

df_stemi = df_icd[df_icd["_is_stemi"]].copy()
df_neg_clear = df_icd[df_icd["_has_any_diag"] & ~df_icd["_has_any_mi"]].copy()

print(f"STEMI dương        : {len(df_stemi):,} bản ghi")
print(f"Non-STEMI (âm rõ)   : {len(df_neg_clear):,} bản ghi")
print(f"Loại (không rõ/MI khác không phải STEMI): "
      f"{len(df_icd) - len(df_stemi) - len(df_neg_clear):,} bản ghi")

if len(df_stemi) == 0:
    print("\n!! KHÔNG tìm thấy ca STEMI nào -- in thử df_icd[DIAG_COL].head(20) để kiểm tra định dạng.")


STEMI dương        : 2,026 bản ghi
Non-STEMI (âm rõ)   : 388,859 bản ghi
Loại (không rõ/MI khác không phải STEMI): 409,150 bản ghi


## 8. Lọc theo bối cảnh lâm sàng (ED/nội trú)

Loại ECG ngoại trú (không trùng đợt khám ED/nội trú nào → không có ngữ cảnh chẩn đoán tin cậy), dùng
cờ `ecg_taken_in_ed_or_hosp` đã xác nhận trong tài liệu.


In [56]:
before_s, before_n = len(df_stemi), len(df_neg_clear)
df_stemi = df_stemi[df_stemi["ecg_taken_in_ed_or_hosp"].astype(bool)]
df_neg_clear = df_neg_clear[df_neg_clear["ecg_taken_in_ed_or_hosp"].astype(bool)]
print(f"Lọc theo ecg_taken_in_ed_or_hosp: STEMI {before_s:,} -> {len(df_stemi):,} | "
      f"Non-STEMI {before_n:,} -> {len(df_neg_clear):,}")


Lọc theo ecg_taken_in_ed_or_hosp: STEMI 2,026 -> 2,026 | Non-STEMI 388,859 -> 388,859


## 9. Lấy mẫu — giữ 100% dương, lấy 10% âm (có chặn trên)


In [57]:
rng = np.random.default_rng(RANDOM_STATE)

n_neg_take = min(int(len(df_neg_clear) * NEG_SAMPLE_FRAC), MAX_NEG_SAMPLES)
df_neg_sample = df_neg_clear.sample(n=n_neg_take, random_state=RANDOM_STATE) if n_neg_take else df_neg_clear.iloc[0:0]

df_sample = pd.concat([
    df_stemi.assign(label=1),
    df_neg_sample.assign(label=0),
], ignore_index=True)

print(f"Mẫu external validation: {len(df_sample):,} bản ghi "
      f"({int(df_sample['label'].sum())} STEMI dương, {int((df_sample['label']==0).sum())} âm)")

assert len(df_sample) > 0, (
    "df_sample RỖNG sau khi lấy mẫu -- df_stemi hoặc df_neg_clear đều không có dòng nào ở mục 7-8. "
    "Kiểm tra lại DIAG_COL/STEMI_CODES và cột ecg_taken_in_ed_or_hosp ở các cell phía trên.")
assert "file_name" in df_sample.columns, (
    f"Không thấy cột 'file_name' trong df_sample -- cột hiện có: {list(df_sample.columns)}. "
    f"Kiểm tra lại tên cột đường dẫn waveform trong records_w_diag_icd10.csv (mục 6).")

display(df_sample[["subject_id", "study_id", "file_name", "label"]].head())


Mẫu external validation: 6,026 bản ghi (2026 STEMI dương, 4000 âm)


,subject_id,study_id,file_name,label
0,10013310,44769849,mimic-iv-ecg-diagnostic-electrocardiogram-matc...,1
1,10013310,46047774,mimic-iv-ecg-diagnostic-electrocardiogram-matc...,1
2,10013310,44367932,mimic-iv-ecg-diagnostic-electrocardiogram-matc...,1
3,10033552,43640229,mimic-iv-ecg-diagnostic-electrocardiogram-matc...,1
4,10033552,40643779,mimic-iv-ecg-diagnostic-electrocardiogram-matc...,1


## 10. Tải waveform (.hea/.dat) — chỉ đúng các bản ghi đã chọn

Cột `file_name` trong bảng Ext-ICD đã cho sẵn đường dẫn waveform trong MIMIC-IV-ECG — không cần tải
thêm `record_list.csv` để merge như cách làm thông thường. Waveform (`MIMIC-IV-ECG` gốc) là **open
access**, không cần xác thực. Tải song song nhẹ để không quá tải server PhysioNet.


In [ ]:
import re
import shutil
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed

MIMIC_BASE = "https://physionet.org/files/mimic-iv-ecg/1.0/"
WFDB_DIR = EXT_WORK_DIR / "wfdb"
WFDB_DIR.mkdir(parents=True, exist_ok=True)
WFDB_CACHE_ZIP = EXT_PERSIST / "mimic_wfdb_cache.zip"   # cache bền trên Drive -- tránh tải lại lần sau

# ---- Nếu đã có cache từ lần chạy trước (lưu trên Drive), giải nén tại chỗ để khỏi tải lại từ mạng ----
if WFDB_CACHE_ZIP.exists():
    print(f"Tìm thấy cache đã tải trước đó: {WFDB_CACHE_ZIP.name} "
          f"({WFDB_CACHE_ZIP.stat().st_size / 1e6:.1f} MB) -- đang giải nén...")
    with zipfile.ZipFile(WFDB_CACHE_ZIP) as zf:
        zf.extractall(WFDB_DIR)
    print(f"Đã giải nén {len(list(WFDB_DIR.glob('*.hea'))):,} bản ghi từ cache "
          f"(vòng tải bên dưới sẽ tự bỏ qua các bản ghi đã có, chỉ tải phần còn thiếu).")

# file_name có thể có/không có đuôi, có/không có tiền tố "files/" tuỳ version, và có thể có thêm
# một tiền tố tên thư mục dataset (vd "mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0/")
# không tồn tại trên URL PhysioNet thật -- chuẩn hoá bằng cách bóc tách đúng phần
# "files/pNNNN/pXXXXXXXX/sZZZZZZZZ/ZZZZZZZZ" ở cuối path.
_sample_fn = str(df_sample["file_name"].iloc[0])
print("Ví dụ file_name gốc:", _sample_fn)

_REL_PATH_RE = re.compile(r"files/p\d{4}/p\d+/s\d+/\d+$")


def _normalize_rel_path(fn: str) -> str:
    fn = str(fn).strip()
    if fn.startswith("http://") or fn.startswith("https://"):
        # phòng trường hợp cột file_name lỡ chứa URL đầy đủ thay vì path tương đối
        fn = fn.split("/files/mimic-iv-ecg/1.0/")[-1]
    fn = fn.lstrip("/")
    for ext in (".hea", ".dat"):
        if fn.endswith(ext):
            fn = fn[: -len(ext)]
    m = _REL_PATH_RE.search(fn)
    if m:
        # bỏ mọi tiền tố thừa (vd tên thư mục dataset) -- chỉ giữ đúng phần path thật trên PhysioNet
        fn = m.group(0)
    elif not fn.startswith("files/"):
        fn = "files/" + fn
    return fn


df_sample["_rel_path"] = df_sample["file_name"].apply(_normalize_rel_path)
_example_rel = df_sample["_rel_path"].iloc[0]
print("Ví dụ rel_path chuẩn hoá:", _example_rel)
print("Ví dụ URL đầy đủ        :", MIMIC_BASE + _example_rel + ".hea")

# ---- Kiểm tra thử 1 URL trước khi tải hàng loạt -- lộ lỗi định dạng sớm, không chờ hết cả batch ----
_probe = requests.get(MIMIC_BASE + _example_rel + ".hea", timeout=30)
print(f"Kiểm tra thử: HTTP {_probe.status_code}"
      + (" -- OK" if _probe.status_code == 200 else " -- LỖI, xem chi tiết bên dưới"))
if _probe.status_code != 200:
    print(_probe.text[:300])
    raise RuntimeError(
        "URL waveform mẫu không tải được (xem HTTP status ở trên). Nguyên nhân thường gặp: cột "
        "file_name không đúng định dạng path tương đối như kỳ vọng ('files/pNNNN/pXXXXXXXX/"
        "sZZZZZZZZ/ZZZZZZZZ'). In df_sample['file_name'].head(10) để xem định dạng thật, rồi sửa "
        "lại hàm _normalize_rel_path cho khớp trước khi chạy tiếp.")


def _download_one(rel_path):
    local_stub = WFDB_DIR / Path(rel_path).name
    for ext in (".hea", ".dat"):
        dst = local_stub.with_suffix(ext)
        if dst.exists():
            continue
        url = MIMIC_BASE + rel_path + ext
        r = requests.get(url, timeout=60)          # open access -- KHÔNG cần auth
        if r.status_code != 200:
            return rel_path, f"HTTP {r.status_code}"
        dst.write_bytes(r.content)
    return rel_path, "ok"


paths = df_sample["_rel_path"].tolist()
t0 = time.time()
results = {}
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = {ex.submit(_download_one, p): p for p in paths}
    for i, fut in enumerate(as_completed(futs), 1):
        rel_path, status = fut.result()
        results[rel_path] = status
        if i % 200 == 0 or i == len(paths):
            print(f"  {i:>5}/{len(paths)}  ({time.time() - t0:.0f}s)")

n_fail = sum(1 for v in results.values() if v != "ok")
print(f"Xong. Lỗi: {n_fail}/{len(paths)}")
if n_fail:
    _fail_list = [k for k, v in results.items() if v != "ok"][:10]
    print("  Vài ví dụ lỗi:", _fail_list)

df_sample["record_stem"] = df_sample["_rel_path"].apply(lambda p: Path(p).name)

# ---- Lưu lại toàn bộ dữ liệu đã tải (kể cả phần giải nén từ cache cũ) lên Drive --------------------
# ---- lần chạy sau sẽ tự động giải nén cache này thay vì tải lại từ PhysioNet -----------------------
print("\nĐang nén dữ liệu waveform để lưu bền vào Drive...")
_tmp_zip = EXT_WORK_DIR / "_wfdb_cache_tmp.zip"
with zipfile.ZipFile(_tmp_zip, "w", zipfile.ZIP_STORED) as zf:
    for f in sorted(WFDB_DIR.glob("*")):
        zf.write(f, arcname=f.name)
shutil.copy(_tmp_zip, WFDB_CACHE_ZIP)
_tmp_zip.unlink()
print(f"Đã lưu cache vào {WFDB_CACHE_ZIP} ({WFDB_CACHE_ZIP.stat().st_size / 1e6:.1f} MB) -- "
      f"lần chạy sau sẽ tự động giải nén cache này thay vì tải lại từ PhysioNet.")

## 11. Tiền xử lý tín hiệu MIMIC-IV-ECG

**Giống hệt notebook 08** (bandpass Butterworth [0,5–40] Hz bậc 3, pad/crop về 5000 mẫu). Có một
điểm **khác** cần xử lý chủ động: MIMIC-IV-ECG không đảm bảo thứ tự 12 chuyển đạo trong file giống
ACS-ECG 2026 — phải đọc `sig_name` trong header và **sắp xếp lại** đúng thứ tự chuẩn
`I, II, III, aVR, aVL, aVF, V1..V6` trước khi đưa vào model, nếu không dữ liệu đưa vào model sẽ bị
lệch kênh một cách âm thầm (không báo lỗi, nhưng dự đoán sai).


In [59]:
import wfdb
from scipy.signal import butter, filtfilt

CANON_LEAD_ORDER = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
_B, _A = butter(BP_ORDER, [BP_LOW / (FS / 2), BP_HIGH / (FS / 2)], btype="band")


def load_and_reorder(stem_path: Path):
    rec = wfdb.rdrecord(str(stem_path))
    idx_map = {}
    for lead in CANON_LEAD_ORDER:
        matches = [i for i, s in enumerate(rec.sig_name)
                  if s.strip().upper() == lead.upper()]
        if not matches:
            return None, f"thiếu lead {lead} (sig_name thực tế: {rec.sig_name})"
        idx_map[lead] = matches[0]
    order = [idx_map[l] for l in CANON_LEAD_ORDER]
    sig = np.asarray(rec.p_signal, dtype=np.float32)[:, order].T   # (12, n_samples)
    return sig, None


def preprocess_ext(sig: np.ndarray) -> np.ndarray:
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    if sig.shape[1] < SIGNAL_LEN:
        sig = np.pad(sig, ((0, 0), (0, SIGNAL_LEN - sig.shape[1])))
    return np.ascontiguousarray(filtfilt(_B, _A, sig[:, :SIGNAL_LEN], axis=1), dtype=np.float32)


EXT_ARR = np.zeros((len(df_sample), NUM_LEADS, SIGNAL_LEN), dtype=np.float32)
EXT_OK = np.zeros(len(df_sample), dtype=bool)
_skip_reasons = []
for i, stem in enumerate(df_sample["record_stem"]):
    hea_path = WFDB_DIR / stem
    try:
        sig, err = load_and_reorder(hea_path)
        if err:
            _skip_reasons.append((stem, err))
            continue
        if sig.shape[1] < FS * 2:              # ECG quá ngắn (< 2s) -- loại
            _skip_reasons.append((stem, f"quá ngắn ({sig.shape[1]} mẫu)"))
            continue
        EXT_ARR[i] = preprocess_ext(sig)
        EXT_OK[i] = True
    except Exception as e:
        _skip_reasons.append((stem, f"{type(e).__name__}: {e}"))

print(f"Tiền xử lý xong: {EXT_OK.sum():,}/{len(df_sample):,} bản ghi hợp lệ")
if _skip_reasons:
    print(f"  {len(_skip_reasons)} bản ghi bị loại, ví dụ:")
    for s, r in _skip_reasons[:10]:
        print(f"    {s}: {r}")

df_sample = df_sample[EXT_OK].reset_index(drop=True)
EXT_ARR = EXT_ARR[EXT_OK]
y_ext = df_sample["label"].values.astype(np.float32)

assert len(df_sample) > 0, (
    "df_sample RỖNG sau tiền xử lý -- TẤT CẢ bản ghi bị loại. Xem lý do cụ thể ở các dòng in phía "
    "trên (thường là thiếu lead do sig_name trong header MIMIC không khớp CANON_LEAD_ORDER, hoặc "
    "file .hea/.dat tải về bị hỏng/thiếu).")

print(f"\nTập external cuối cùng: {len(df_sample):,} bản ghi "
      f"({int(y_ext.sum())} STEMI dương, {y_ext.mean():.2%})")


Tiền xử lý xong: 6,025/6,026 bản ghi hợp lệ
  1 bản ghi bị loại, ví dụ:
    44900990: FileNotFoundError: [Errno 2] No such file or directory: '/content/mimic_ext/wfdb/44900990.hea'

Tập external cuối cùng: 6,025 bản ghi (2026 STEMI dương, 33.63%)


## 12. Nạp 8 checkpoint FINAL và dự đoán ensemble Average

Dùng đúng checkpoint `{tên}_FINAL_pool.pt` (train 1 lần trên 100% POOL, mục 14c notebook 08) —
đây là checkpoint sản xuất, không phải các checkpoint theo fold.


In [60]:
from torch.utils.data import DataLoader, Dataset


class ExtDataset(Dataset):
    def __init__(self, arr, labels, mean, std):
        self.arr = arr
        self.labels = np.asarray(labels, dtype=np.float32)
        self.mean = mean.reshape(-1, 1)
        self.std = std.reshape(-1, 1)

    def __len__(self):
        return len(self.arr)

    def __getitem__(self, i):
        x = self.arr[i]
        return torch.from_numpy((x - self.mean) / self.std), torch.tensor(self.labels[i])


ext_loader = DataLoader(ExtDataset(EXT_ARR, y_ext, mean_pool, std_pool),
                        batch_size=BATCH_SIZE, shuffle=False)


def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            ys.append(yb.numpy())
            ps.append(torch.sigmoid(logits.float()).cpu().numpy())
    y_out, p_out = np.concatenate(ys), np.concatenate(ps)
    bad = ~np.isfinite(p_out)
    if bad.any():
        p_out = np.where(bad, 0.5, p_out)
    return y_out, p_out


EXT_P = {}
for name in CANDIDATES:
    ckpt_path = KFOLD_MODEL_DIR / f"{name.replace('+', '_')}_FINAL_pool.pt"
    assert ckpt_path.exists(), f"Không tìm thấy checkpoint {ckpt_path}"
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model = MODELS[name]().to(DEVICE)
    model.load_state_dict(ck["model"])
    _, p = evaluate(model, ext_loader)
    EXT_P[name] = p
    del model
    if GPU_AVAILABLE:
        torch.cuda.empty_cache()
    print(f"  {name}: xong")

EXT_P["Average Ensemble"] = np.mean([EXT_P[n] for n in CANDIDATES], axis=0)
print("\nĐã dự đoán xong 8 model đơn + Average Ensemble trên tập external.")


  PlainCNN: xong
  XResNet1D: xong
  ConvNeXtV2_1D: xong
  AiTiAMI: xong
  ResNet1D: xong
  SEResNet1D: xong
  CNN+BiLSTM: xong
  InceptionTime1D: xong

Đã dự đoán xong 8 model đơn + Average Ensemble trên tập external.


## 13. Áp ngưỡng đã chốt từ POOL — tính metric đầy đủ trên MIMIC-IV-ECG

**Không refit ngưỡng trên MIMIC** — dùng nguyên `THRESHOLD` đã tính ở mục 6 (từ OOF trên ACS-ECG
2026), đúng phương pháp luận notebook 08.


In [61]:
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             roc_auc_score)


def _div(a, b):
    return float(a) / float(b) if b else float("nan")


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=np.float64).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    two = len(np.unique(y_true)) == 2
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "auroc": roc_auc_score(y_true, y_prob) if two else float("nan"),
        "auprc": average_precision_score(y_true, y_prob) if two else float("nan"),
        "sensitivity": _div(tp, tp + fn), "specificity": _div(tn, tn + fp),
        "ppv": _div(tp, tp + fp), "npv": _div(tn, tn + fn),
        "f1": _div(2 * tp, 2 * tp + fp + fn),
        "brier": float(brier_score_loss(y_true, y_prob)) if two else float("nan"),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "n": int(len(y_true)), "n_pos": int(y_true.sum()),
    }


rows = []
for name in CANDIDATES + ["Average Ensemble"]:
    m = compute_metrics(y_ext, EXT_P[name], THRESHOLD if name == "Average Ensemble" else 0.5)
    rows.append({"Model": name, **m})
ext_results_df = pd.DataFrame(rows).set_index("Model")

print("=" * 78)
print(f"EXTERNAL VALIDATION -- MIMIC-IV-ECG (STEMI-proxy qua ICD-10), {len(df_sample):,} bản ghi")
print(f"Ngưỡng Average Ensemble @ Sensitivity >= {TARGET_SENS_PRIMARY:.0%} (chốt từ ACS-ECG 2026 POOL, "
      f"KHÔNG refit): {THRESHOLD:.4f}")
print("=" * 78)
display(ext_results_df[["auroc", "auprc", "sensitivity", "specificity", "ppv", "npv", "f1",
                        "n", "n_pos"]].round(4))

_champ = ext_results_df.loc["Average Ensemble"]
print(f"\nAverage Ensemble trên MIMIC-IV-ECG: AUROC {_champ['auroc']:.4f} | "
      f"Sensitivity {_champ['sensitivity']:.4f} | Specificity {_champ['specificity']:.4f} | "
      f"NPV {_champ['npv']:.4f} | PPV {_champ['ppv']:.4f}")


EXTERNAL VALIDATION -- MIMIC-IV-ECG (STEMI-proxy qua ICD-10), 6,025 bản ghi
Ngưỡng Average Ensemble @ Sensitivity >= 91% (chốt từ ACS-ECG 2026 POOL, KHÔNG refit): 0.2417


,auroc,auprc,sensitivity,specificity,ppv,npv,f1,n,n_pos
Model,,,,,,,,,
PlainCNN,0.8291,0.7478,0.6407,0.8510,0.6853,0.8238,0.6622,6025,2026
XResNet1D,0.8230,0.7354,0.4867,0.9277,0.7733,0.7811,0.5974,6025,2026
ConvNeXtV2_1D,0.8272,0.7492,0.5192,0.9242,0.7764,0.7914,0.6223,6025,2026
AiTiAMI,0.8332,0.7420,0.5005,0.9187,0.7573,0.7840,0.6027,6025,2026
ResNet1D,0.8414,0.7626,0.6106,0.8847,0.7285,0.8177,0.6643,6025,2026
SEResNet1D,0.8349,0.7576,0.5859,0.8957,0.7400,0.8102,0.6540,6025,2026
CNN+BiLSTM,0.8359,0.7532,0.5602,0.9020,0.7433,0.8019,0.6389,6025,2026
InceptionTime1D,0.8269,0.7459,0.4886,0.9392,0.8029,0.7838,0.6075,6025,2026
Average Ensemble,0.8523,0.7833,0.7285,0.8255,0.6789,0.8572,0.7029,6025,2026



Average Ensemble trên MIMIC-IV-ECG: AUROC 0.8523 | Sensitivity 0.7285 | Specificity 0.8255 | NPV 0.8572 | PPV 0.6789


## 13b. Confusion Matrix


In [ ]:
import matplotlib.pyplot as plt


def plot_confusion(name, ax):
    row = ext_results_df.loc[name]
    cm = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]], dtype=int)
    ax.imshow(cm, cmap="Blues")
    for r in range(2):
        for c in range(2):
            ax.text(c, r, f"{cm[r, c]:,}", ha="center", va="center",
                    color="white" if cm[r, c] > cm.max() / 2 else "black", fontsize=13)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Dự đoán Âm", "Dự đoán Dương"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Thực Âm", "Thực Dương"])
    thr = THRESHOLD if name == "Average Ensemble" else 0.5
    ax.set_title(f"{name}\n(ngưỡng={thr:.4f})", fontsize=11)


fig, ax = plt.subplots(figsize=(4.5, 4.5))
plot_confusion("Average Ensemble", ax)
plt.tight_layout()
plt.show()

_cm_champ = ext_results_df.loc["Average Ensemble", ["tn", "fp", "fn", "tp"]]
print(f"Confusion matrix -- Average Ensemble (ngưỡng {THRESHOLD:.4f}, "
      f"Sensitivity mục tiêu >= {TARGET_SENS_PRIMARY:.0%}):")
print(f"  TN={_cm_champ['tn']:,}  FP={_cm_champ['fp']:,}")
print(f"  FN={_cm_champ['fn']:,}  TP={_cm_champ['tp']:,}")

print(f"\nConfusion matrix đầy đủ -- 8 model đơn (ngưỡng 0.5) + Average Ensemble "
      f"(ngưỡng {THRESHOLD:.4f}):")
display(ext_results_df[["tn", "fp", "fn", "tp", "n", "n_pos"]])

## 14. So sánh với kết quả nội bộ (TEST giữ riêng, ACS-ECG 2026)

Đặt cạnh nhau để thấy mức độ suy giảm (nếu có) khi ra khỏi phân bố đã học.


In [62]:
# Dán thủ công kết quả Average Ensemble trên TEST nội bộ từ notebook 08 (mục 21d) nếu có sẵn,
# hoặc để trống -- so sánh bằng mắt qua 2 bảng.
print("Đối chiếu:")
print(f"  MIMIC-IV-ECG (external, {len(df_sample):,} bản ghi, {int(y_ext.sum())} dương):")
print(f"    AUROC {_champ['auroc']:.4f}  Sensitivity {_champ['sensitivity']:.4f}  "
      f"Specificity {_champ['specificity']:.4f}  NPV {_champ['npv']:.4f}  PPV {_champ['ppv']:.4f}")
print("  ACS-ECG 2026 TEST giữ riêng (notebook 08, mục 21d): điền thủ công từ output notebook 08 để so sánh.")


Đối chiếu:
  MIMIC-IV-ECG (external, 6,025 bản ghi, 2026 dương):
    AUROC 0.8523  Sensitivity 0.7285  Specificity 0.8255  NPV 0.8572  PPV 0.6789
  ACS-ECG 2026 TEST giữ riêng (notebook 08, mục 21d): điền thủ công từ output notebook 08 để so sánh.


## 15. Lưu kết quả về Drive


In [63]:
import json as _json

ext_results_df.to_csv(EXT_PERSIST / "external_validation_mimic_results.csv")
np.savez_compressed(EXT_PERSIST / "external_validation_mimic_predictions.npz",
                    y=y_ext, subject_id=df_sample["subject_id"].to_numpy(),
                    study_id=df_sample["study_id"].to_numpy(),
                    **{f"p_{n}": EXT_P[n] for n in CANDIDATES},
                    p_average=EXT_P["Average Ensemble"])
(EXT_PERSIST / "run_meta.json").write_text(_json.dumps({
    "threshold": THRESHOLD, "target_sensitivity": TARGET_SENS_PRIMARY,
    "n_total": len(df_sample), "n_pos": int(y_ext.sum()),
    "neg_sample_frac": NEG_SAMPLE_FRAC, "stemi_codes": sorted(STEMI_CODES),
    "diag_col": DIAG_COL,
}, indent=2))
print("Đã lưu:", EXT_PERSIST)


Đã lưu: /content/drive/MyDrive/ACS-ECG-AI/outputs/external_validation/mimic_icd


## 16. Hạn chế cần nêu khi báo cáo kết quả này

- **Nhãn là STEMI-proxy qua ICD-10**, không phải chẩn đoán ECG-tại-thời-điểm được bác sĩ tim mạch xác
  nhận như ACS-ECG 2026 (angio-confirmed). Sai số nhãn có thể làm cả AUROC lẫn Sensitivity/Specificity
  bị lệch theo hướng khó dự đoán trước.
- **Không đảm bảo ECG lấy đúng thời điểm biến cố STEMI** — mã ICD chỉ phản ánh chẩn đoán ra viện của
  cả đợt điều trị, ECG có thể ghi trước/sau thời điểm ST chênh lên thực sự khá xa.
- **Lấy mẫu không ngẫu nhiên hoàn toàn** (giữ 100% dương, lấy 10% âm) — không dùng con số AUROC/AUPRC
  ở đây để suy ra prevalence hoặc so sánh trực tiếp base rate với ACS-ECG 2026.
- **Domain gap**: dân số Mỹ (BIDMC Boston) vs Việt Nam, thiết bị ghi khác nhau (Burdick/Spacelabs,
  Philips, GE vs thiết bị ACS-ECG 2026) — chênh lệch hiệu năng một phần phản ánh domain shift, không
  chỉ chất lượng model.
- Kết quả ở đây nên được trình bày như **kiểm chứng bổ sung mang tính khám phá (exploratory)**, đặt
  cạnh — không thay thế — kết quả TEST nội bộ của notebook 08.
